In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

In [ ]:
BASE = Path("..")

TRAIN_DIR = BASE / "dataset" / "train"
TEST_DIR = BASE / "dataset" / "test"

print(TRAIN_DIR)
print(TEST_DIR)

In [ ]:
files = [
    TRAIN_DIR / "train_source1.tsv",
    TRAIN_DIR / "train_source2.tsv",
    TRAIN_DIR / "train_source3.tsv",
    TRAIN_DIR / "train_ground_truth.tsv",
    TEST_DIR / "test_source1.tsv",
    TEST_DIR / "test_source2.tsv",
    TEST_DIR / "test_source3.tsv",
]

for f in files:
    size_mb = f.stat().st_size / (1024 ** 2)
    print(f"{f.name:25} {size_mb:,.1f} MB")

In [ ]:
#read a small sample

#This is the first important Python operation.
s1_sample = pd.read_csv(
    TRAIN_DIR / "train_source1.tsv",
    sep="\t",
    nrows=10_000
)

s2_sample = pd.read_csv(
    TRAIN_DIR / "train_source2.tsv",
    sep="\t",
    nrows=10_000
)

s3_sample = pd.read_csv(
    TRAIN_DIR / "train_source3.tsv",
    sep="\t",
    nrows=10_000
)

gt_sample = pd.read_csv(
    TRAIN_DIR / "train_ground_truth.tsv",
    sep="\t",
    nrows=10_000
)

In [ ]:
s1_sample = pd.read_csv(
    TRAIN_DIR / "train_source1.tsv",
    sep="\t",
    nrows=10_000
)

s2_sample = pd.read_csv(
    TRAIN_DIR / "train_source2.tsv",
    sep="\t",
    nrows=10_000
)

s3_sample = pd.read_csv(
    TRAIN_DIR / "train_source3.tsv",
    sep="\t",
    nrows=10_000
)

gt_sample = pd.read_csv(
    TRAIN_DIR / "train_ground_truth.tsv",
    sep="\t",
    nrows=10_000
)

In [ ]:
print("S1:", s1_sample.shape)
print("S2:", s2_sample.shape)
print("S3:", s3_sample.shape)
print("GT:", gt_sample.shape)

In [ ]:
for name, df in [
    ("S1", s1_sample),
    ("S2", s2_sample),
    ("S3", s3_sample),
    ("Ground Truth", gt_sample),
]:
    print(f"\n{name}")
    print(df.columns.tolist())

In [ ]:
for name, df in [
    ("S1", s1_sample),
    ("S2", s2_sample),
    ("S3", s3_sample),
    ("Ground Truth", gt_sample),
]:
    print(f"\n{name}")
    print(df.isna().sum())

In [ ]:
match_counts = []

for chunk in pd.read_csv(
    TRAIN_DIR / "train_ground_truth.tsv",
    sep="\t",
    chunksize=100_000
):
    counts = (
        chunk["matched_entity_ids"]
        .fillna("")
        .apply(
            lambda x: 0 if x == "" else len(x.split(","))
        )
    )

    match_counts.append(counts)

match_counts = pd.concat(match_counts, ignore_index=True)

In [ ]:
match_distribution = match_counts.value_counts().sort_index()

print(match_distribution)

In [ ]:
print("Average matches per S1:", match_counts.mean())
print("Maximum matches for one S1:", match_counts.max())
print("Zero-match S1 count:", (match_counts == 0).sum())
print(
    "Zero-match percentage:",
    (match_counts == 0).mean() * 100
)

In [ ]:
for name, df in [
    ("S1", s1_sample),
    ("S2", s2_sample),
    ("S3", s3_sample)
]:
    print(f"\n{name} countries:")
    print(df["country"].value_counts(dropna=False).head(20))

In [ ]:
for name, df in [
    ("S1", s1_sample),
    ("S2", s2_sample),
    ("S3", s3_sample)
]:
    print(f"\n{name} address lengths:")
    print(df["business_address"].fillna("").str.len().describe())

    print(f"\n{name} name lengths:")
    print(df["business_name"].fillna("").str.len().describe())

In [ ]:
import re

def normalize_text(text):
    if pd.isna(text):
        return ""
    
    text = str(text).lower()
    text = re.sub(r"[^a-z0-9\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    
    return text

In [ ]:
test_names = [
    "ABC Technologies Pvt. Ltd.",
    "McDonald's Restaurant",
    "  Prime   Money  ",
    "B+ Retail Inc."
]

for x in test_names:
    print(x, "→", normalize_text(x))

In [ ]:
# What we established
# Dataset sizes and structure — S1, S2, S3, and ground truth.
# Scale of the problem — roughly 22.8 trillion possible S1 ↔ S2/S3 comparisons, proving brute force is infeasible.
# Hardware constraints — 16 GB RAM, so we should use samples/chunks rather than loading everything into pandas.
# Schema — entity_id, business_name, business_address, country.
# Missing values — addresses are occasionally missing; names/countries were complete in the inspected samples.
# Match cardinality — S1 entities can have 0–11 matches.
# Average matches — about 3.46 per S1.
# Zero-match cases — 123,247 / 2,206,821 ≈ 5.58%.
# Country distribution — only US and India in our samples, with similar distributions across sources.
# Name/address length distributions — broadly similar across sources.
# Initial normalization function — implemented and working.